# SBBTS — Showcase Notebook

End-to-end demonstration of the SBBTS library for synthetic financial time series generation.

**What this notebook covers:**
1. Loading SPX data and building rolling windows
2. Training SBBTS (Lite config for speed)
3. `compute_metrics()` — stylized-facts comparison dict
4. `log_ret_to_prices()` — reconstruct price paths from log returns
5. `sample_conditional()` — fan charts: generate continuations of a real prefix
6. `sample_batches()` — memory-safe large-scale generation
7. `compute_tstr()` — Train-on-Synthetic, Test-on-Real quality score
8. `augment()` + `save()` / `load()` — data augmentation pipeline
9. `fit(resume_from_step=k)` — warm restart from checkpoint

Based on: *SBBTS: A Unified Schrödinger–Bass Framework for Synthetic Financial Time Series*, Alouadi et al. (2026)

## 0 — Setup & global config

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                            'grid.alpha': 0.3, 'axes.spines.top': False,
                            'axes.spines.right': False})

from sbbts import (
    SBBTS,
    compute_metrics, compute_tstr, log_ret_to_prices,
)

# ── Global config ──────────────────────────────────────────────────────
N_DAYS = 500   # close prices to keep (tail); 500 ≈ 2 yrs, 1260 ≈ 5 yrs
T      = 252   # window length in time steps (252 = 1 trading year)
# ───────────────────────────────────────────────────────────────────────

print(f"N_DAYS = {N_DAYS}  ({N_DAYS/252:.1f} trading years)")
print(f"T      = {T}  (window length)")
print(f"Windows available: {N_DAYS - T}")

## 1 — Data: SPX log returns → rolling windows

We use `yfinance` to pull SPX close prices.  
If yfinance is unavailable, the cell falls back to a Geometric Brownian Motion simulation so the rest of the notebook still runs.

In [ ]:
try:
    import yfinance as yf
    prices = yf.download("^GSPC", period="20y", auto_adjust=True, progress=False)["Close"]
    prices = prices.dropna().values[-N_DAYS:]
    print(f"SPX loaded: {len(prices)} days")
except Exception:
    # Fallback: simulate GBM prices
    print("yfinance unavailable — using GBM simulation")
    np.random.seed(42)
    daily_vol = 0.012
    prices = 4000.0 * np.exp(np.cumsum(
        np.random.normal(0, daily_vol, N_DAYS)
    ))

# Log returns: shape (N_DAYS - 1,)
log_ret = np.log(prices[1:] / prices[:-1]).astype(np.float32)

# Rolling windows of length T: shape (N_windows, T, 1)
windows = np.lib.stride_tricks.sliding_window_view(log_ret, T)
X_train = windows[:, :, np.newaxis]           # (N_windows, T, 1)

print(f"Log returns — mean: {log_ret.mean():.5f}  std: {log_ret.std():.5f}")
print(f"X_train shape: {X_train.shape}  ({X_train.shape[0]} windows)")

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
axes[0].plot(prices, lw=0.8, color='#2563eb')
axes[0].set_title("SPX price (tail)")
axes[1].plot(log_ret, lw=0.5, color='#dc2626', alpha=0.8)
axes[1].set_title("SPX log returns")
plt.tight_layout()
plt.show()

## 2 — Choose β and train

`suggest_beta()` computes the smallest β satisfying `β·Δt = safety_factor` (Theorem 3.2 requires `β·Δt > 1`).  
We use the **Lite config** here so training takes 10–20 min on CPU.  Swap in `FULL_CFG` for paper-quality results.

In [ ]:
BETA = SBBTS.suggest_beta(n_time_points=T, safety_factor=5.0)
print(f"β = {BETA:.1f}  (β·Δt = {BETA/(T-1):.2f}, threshold = 1.0)")

LITE_CFG = dict(
    beta              = BETA,
    n_steps           = 2,       # paper: 5
    d_model           = 32,      # paper: 128
    n_heads           = 4,       # paper: 16
    n_encoder_layers  = 1,
    n_epochs          = 300,     # paper: 1000
    batch_size        = 128,
    learning_rate     = 1e-3,
    n_euler_steps     = 20,      # paper: 50
    normalize_input   = True,
    grad_clip         = 0.0,
)

FULL_CFG = dict(
    beta              = BETA,
    n_steps           = 5,
    d_model           = 128,
    n_heads           = 16,
    n_encoder_layers  = 1,
    n_epochs          = 1000,
    batch_size        = 128,
    learning_rate     = 3e-4,
    n_euler_steps     = 50,
    normalize_input   = True,
    grad_clip         = 0.0,
)

# Switch to FULL_CFG for paper-quality results (needs ~8 hrs on CPU, ~15 min on A100)
RUN_FULL = False
cfg = FULL_CFG if RUN_FULL else LITE_CFG

model = SBBTS(**cfg)
model.fit(X_train, verbose=True)
print("Training complete.")

## 3 — Basic sampling + `compute_metrics()`

`compute_metrics()` returns a plain dict comparing stylized facts between real and synthetic data — no matplotlib required.  
Use it for logging, CI assertions, or automated hyperparameter search.

In [ ]:
X_synth = model.sample(n=500)   # (500, T, 1)

m = compute_metrics(X_train, X_synth)

# Print a tidy table of key metrics
rows = [
    ("ann_std",          "Annualised vol",       m["ann_std_real"],         m["ann_std_synth"],         m["ann_std_ratio"]),
    ("rv_mean",          "RV mean",               m["rv_mean_real"],          m["rv_mean_synth"],          m["rv_mean_ratio"]),
    ("rv_std",           "RV std",                m["rv_std_real"],           m["rv_std_synth"],           m["rv_std_ratio"]),
    ("kurtosis",         "Excess kurtosis",       m["kurtosis_real"],         m["kurtosis_synth"],         m["kurtosis_ratio"]),
    ("acf_abs_sum",      "ACF |r| (lags 1-5)",   m["acf_abs_sum_real"],      m["acf_abs_sum_synth"],      m["acf_abs_sum_ratio"]),
    ("leverage_k1",      "Leverage k=1",          m["leverage_k1_real"],      m["leverage_k1_synth"],      None),
    ("rolling_vol_mean", "Rolling vol mean",      m["rolling_vol_mean_real"], m["rolling_vol_mean_synth"], m["rolling_vol_mean_ratio"]),
]

print(f"{'Metric':<22} {'Real':>10} {'SBBTS':>10} {'Ratio':>8}")
print("-" * 54)
for _, label, real, synth, ratio in rows:
    ratio_str = f"{ratio:.3f}" if ratio is not None else "—"
    print(f"{label:<22} {real:>10.4f} {synth:>10.4f} {ratio_str:>8}")

print()
print("Target ratios: 0.8–1.2 for vol metrics, leverage sign should match")

## 4 — `log_ret_to_prices()` — price path reconstruction

Convert synthetic log return windows back to price paths via `S_t = S_0 · exp(Σ r_k)`.  
Useful for realistic P&L simulation or option pricing applications.

In [ ]:
S0 = prices[-T - 1]   # last real price before the final window starts

# Reconstruct price paths from synthetic log returns
# X_synth: (500, T, 1) → prices_synth: (500, T+1, 1)
prices_synth = log_ret_to_prices(X_synth, S0=S0)
prices_real  = log_ret_to_prices(X_train[-1:], S0=S0)  # last real window

print(f"prices_synth shape: {prices_synth.shape}")

fig, ax = plt.subplots(figsize=(12, 4))
t_axis = np.arange(T + 1)

# Plot 40 synthetic paths
for i in range(40):
    ax.plot(t_axis, prices_synth[i, :, 0], color='#93c5fd', lw=0.5, alpha=0.6)

# Plot real path on top
ax.plot(t_axis, prices_real[0, :, 0], color='#1e3a5f', lw=2.0, label='Real', zorder=5)

ax.set_title(f"Price paths — 40 synthetic vs 1 real  (S₀ = {S0:.0f})")
ax.set_xlabel("Trading days")
ax.set_ylabel("Price")
ax.legend()
plt.tight_layout()
plt.show()

## 5 — `sample_conditional()` — fan chart

Given an observed prefix of `T_prefix` days, generate `n` plausible continuations.  
Applications: conditional stress testing, scenario analysis, fan chart visualization.

The prefix can be any length from 1 to T-1.

In [ ]:
# Use the last 50 days of the final real window as the conditioning prefix
T_prefix = 50
X_prefix = X_train[-1, :T_prefix, :]   # (T_prefix, 1) — last real window, first T_prefix days

# Generate 200 continuations of the remaining T - T_prefix days
X_fan = model.sample_conditional(X_prefix, n=200)   # (200, T, 1)

print(f"Fan shape: {X_fan.shape}")
print(f"Prefix: days 0–{T_prefix-1}  |  Generated: days {T_prefix}–{T-1}")

# Convert to cumulative return for easier visual interpretation
cum_ret_fan  = np.cumsum(X_fan[:, :, 0], axis=1)   # (200, T)
cum_ret_real = np.cumsum(X_train[-1, :, 0])         # (T,)

fig, ax = plt.subplots(figsize=(12, 4))
t = np.arange(T)

# Fan: median + IQR + 5th/95th percentile bands
p05 = np.percentile(cum_ret_fan, 5,  axis=0)
p25 = np.percentile(cum_ret_fan, 25, axis=0)
p50 = np.percentile(cum_ret_fan, 50, axis=0)
p75 = np.percentile(cum_ret_fan, 75, axis=0)
p95 = np.percentile(cum_ret_fan, 95, axis=0)

ax.fill_between(t[T_prefix:], p05[T_prefix:], p95[T_prefix:],
                color='#93c5fd', alpha=0.25, label='5–95th pct')
ax.fill_between(t[T_prefix:], p25[T_prefix:], p75[T_prefix:],
                color='#3b82f6', alpha=0.35, label='IQR')
ax.plot(t[T_prefix:], p50[T_prefix:], color='#1d4ed8', lw=1.5, label='Median')

# Prefix (observed)
ax.plot(t[:T_prefix], cum_ret_real[:T_prefix], color='#1e3a5f', lw=2.0, label='Observed prefix')

# Full real path as reference
ax.plot(t, cum_ret_real, color='#dc2626', lw=1.5, ls='--', alpha=0.6, label='Real (full)')

ax.axvline(T_prefix - 1, color='gray', ls=':', lw=1)
ax.set_title(f"Conditional fan chart — prefix = {T_prefix} days, n = 200 continuations")
ax.set_xlabel("Trading day")
ax.set_ylabel("Cumulative log return")
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

# Fan spread statistics
fan_std = X_fan[:, -1, 0].std()
real_final = float(X_train[-1, -1, 0])
real_in_iqr = p25[-1] <= cum_ret_real[-1] <= p75[-1]
print(f"Fan std at final step : {fan_std:.5f}")
print(f"Real final value      : {real_final:.5f}")
print(f"Real in fan IQR       : {real_in_iqr}")

## 6 — `sample_batches()` — memory-safe large generation

When generating 10 000+ paths, `sample(n=50000)` may OOM.  
`sample_batches()` is a generator that yields batches so memory stays bounded.

In [ ]:
from tqdm.auto import tqdm

N_LARGE  = 2000    # total synthetic paths to generate
BSIZE    = 200     # max paths per batch (tune to available RAM)

chunks = []
for batch in tqdm(model.sample_batches(n=N_LARGE, batch_size=BSIZE),
                  total=N_LARGE // BSIZE, desc="Generating"):
    chunks.append(batch)

X_large = np.concatenate(chunks, axis=0)
print(f"Generated {X_large.shape[0]} synthetic windows  shape={X_large.shape}")
print(f"Memory: {X_large.nbytes / 1e6:.1f} MB")

## 7 — `compute_tstr()` — downstream quality evaluation

**TSTR (Train-on-Synthetic, Test-on-Real):** fits a linear AR(5) on synthetic data,
evaluates its MSE on a held-out real test set, then compares to **TRTR** (same model fit on real training data).

- Ratio ≈ 1.0 → synthetic is a near-perfect substitute  
- Ratio < 1.05 → excellent  
- Ratio > 1.5 → model has not captured the predictive structure

In [ ]:
tstr = compute_tstr(X_train, X_large, ar_order=5, test_fraction=0.2)

print("TSTR Evaluation — AR(5) on SPX windows")
print("=" * 45)
print(f"  TRTR MSE (train=real,  test=real)  : {tstr['trtr_mse']:.8f}")
print(f"  TSTR MSE (train=synth, test=real)  : {tstr['tstr_mse']:.8f}")
print(f"  Ratio (TSTR / TRTR)                : {tstr['ratio']:.4f}")
print()
print(f"  Real train samples  : {tstr['n_real_train']}")
print(f"  Real test  samples  : {tstr['n_real_test']}")
print(f"  Synth train samples : {tstr['n_synth_train']}")
print()
if tstr['ratio'] < 1.05:
    print("✓ Synthetic is an excellent substitute (ratio < 1.05)")
elif tstr['ratio'] < 1.2:
    print("~ Synthetic is a reasonable substitute (ratio < 1.2)")
else:
    print("✗ Synthetic quality is limited — consider more data or FULL config")

## 8 — Data augmentation pipeline

The primary use case: generate `factor × N` synthetic windows and concatenate with real data
to improve a downstream forecasting model.

In [ ]:
# Augment: real data + 10× synthetic  (use factor=200 for paper experiments)
X_aug = model.augment(X_train, factor=10)

print(f"Real windows         : {X_train.shape[0]}")
print(f"Augmented dataset    : {X_aug.shape[0]}  (+{X_aug.shape[0] - X_train.shape[0]} synthetic)")
print(f"X_aug shape          : {X_aug.shape}")

## 9 — Save, load, and warm restart

Training takes hours for the full config.  
Save after every outer step and use `resume_from_step=k` to continue from a checkpoint.

In [ ]:
# Save the current model
model.save("sbbts_lite.pt")
print("Model saved to sbbts_lite.pt")

# Load it back
model_loaded = SBBTS.load("sbbts_lite.pt")
X_check = model_loaded.sample(n=5)
print(f"Loaded model sample shape: {X_check.shape}")

# Warm restart: suppose this was a FULL_CFG model that completed k=1 and k=2,
# then the process was interrupted.  After loading:
#
#   model_resumed = SBBTS.load("sbbts_checkpoint_k2.pt", **FULL_CFG)
#   model_resumed.fit(X_train, resume_from_step=2)  # starts at k=3, keeps learned weights
#
# Note: the FULL_CFG parameters must match the original run.
print()
print("Warm restart example (not run here — needs a multi-step checkpoint):")
print("  model_resumed = SBBTS.load('checkpoint_k2.pt')")
print("  model_resumed.fit(X_train, resume_from_step=2)")

## 10 — Summary

| Feature | Function | Returns |
|---|---|---|
| Train | `model.fit(X)` | self |
| Basic sampling | `model.sample(n)` | `(n, T, d)` ndarray |
| Conditional fan | `model.sample_conditional(X_prefix, n)` | `(n, T, d)` ndarray |
| Memory-safe generation | `model.sample_batches(n, batch_size)` | generator of batches |
| Data augmentation | `model.augment(X_real, factor)` | `((1+factor)·N, T, d)` |
| Stylized facts | `compute_metrics(X_real, X_synth)` | dict |
| TSTR evaluation | `compute_tstr(X_real, X_synth)` | dict with ratio |
| Price reconstruction | `log_ret_to_prices(X, S0)` | `(N, T+1, d)` ndarray |
| Save / load | `model.save(path)` / `SBBTS.load(path)` | — |
| Warm restart | `model.fit(X, resume_from_step=k)` | self |

See [README.md](README.md) for the full hyperparameter reference and [THEORY.md](THEORY.md) for the mathematical derivation.